# 05 — Reflect Deep Dive: Disposition-Aware Reasoning

This notebook covers `reflect()` in depth:
- Recall vs Reflect: when to use each
- Budget levels and token control
- Structured output with `response_schema`
- Context parameter for additional instructions
- How disposition traits change reasoning
- Directives: hard guardrails
- Evidence tracking with `based_on`

In [ ]:
from hindsight_client import Hindsight
from datetime import datetime, timezone
import json

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-reflect"

client = Hindsight(base_url=HINDSIGHT_URL)

# Create a bank with personality
client.create_bank(
    bank_id=BANK_ID,
    name="Reflect Tutorial",
    reflect_mission="""
    I am a technical project manager assistant.
    I track project status, team decisions, and risks.
    I prioritize practical, data-driven recommendations.
    I'm thorough but concise in my analysis.
    """,
    disposition={"skepticism": 3, "literalism": 3, "empathy": 2}
)

# Seed with project data
project_data = [
    # Project info
    {"content": "Project Phoenix: migrating legacy monolith to microservices", "context": "project"},
    {"content": "Current monolith is 500k lines of Java, deployed on-prem", "context": "current-state"},
    {"content": "Target: 12 microservices on AWS EKS, Go + Python", "context": "target"},
    {"content": "Team size: 15 engineers, split into 3 squads", "context": "team"},
    
    # Progress and timeline
    {"content": "Phase 1 complete: User service extracted (March 2026)", "context": "progress",
     "timestamp": "2026-03-15T00:00:00Z"},
    {"content": "Phase 2 in progress: Order service migration, 60% done", "context": "progress",
     "timestamp": "2026-05-01T00:00:00Z"},
    {"content": "Phase 2 behind schedule by 3 weeks due to data consistency issues", "context": "blocker",
     "timestamp": "2026-06-01T00:00:00Z"},
    
    # Risks and issues
    {"content": "Risk: shared database has tight coupling between services", "context": "risk"},
    {"content": "Risk: on-prem network latency to AWS could affect migration", "context": "risk"},
    {"content": "Issue: test coverage is 45%, should be 80% before production cutover", "context": "issue"},
    {"content": "Issue: 2 senior engineers left the team in Q1", "context": "issue"},
    
    # Decisions
    {"content": "Decided to use event sourcing pattern for order service", "context": "decision"},
    {"content": "Chose NATS over Kafka for inter-service messaging (simpler ops)", "context": "decision"},
    {"content": "Timeline: aiming for full migration by September 2026", "context": "plan"},
]

for item in project_data:
    ts = item.pop("timestamp", None)
    client.retain(bank_id=BANK_ID, timestamp=ts, **item)

print(f"✓ Seeded {len(project_data)} project memories")

## 1. Basic Reflect — Ask a Question, Get an Answer

In [ ]:
# Ask Hindsight to reason about the project
answer = client.reflect(
    bank_id=BANK_ID,
    query="What is the current status of Project Phoenix and what are the key risks?"
)

print("=== Hindsight's Analysis ===")
print(answer.text)
print(f"\n---")
print(f"Token usage: {answer.usage.total_tokens} total")

## 2. Evidence Tracking — Where Did That Come From?

Every reflect response tracks its sources with `based_on`.

In [ ]:
answer = client.reflect(
    bank_id=BANK_ID,
    query="What is the biggest concern for the September deadline?",
    budget="high"
)

print("=== Answer ===")
print(answer.text)

print(f"\n=== Evidence ({len(answer.based_on)} sources) ===")
for i, source in enumerate(answer.based_on, 1):
    print(f"{i}. [{source.type}] {source.text[:120]}...")

## 3. Budget Levels

In [ ]:
# Compare budgets for the same question
for budget in ["low", "mid", "high"]:
    answer = client.reflect(
        bank_id=BANK_ID,
        query="Summarize the project status in one sentence.",
        budget=budget
    )
    print(f"\nBudget: {budget}")
    print(f"  Answer: {answer.text[:200]}")
    print(f"  Sources: {len(answer.based_on)}")

## 4. Structured Output with response_schema

Request JSON output with a specific schema for programmatic consumption.

In [ ]:
# Define a schema for a risk assessment report
risk_schema = {
    "type": "object",
    "properties": {
        "project_status": {
            "type": "string",
            "enum": ["on_track", "at_risk", "critical"]
        },
        "summary": {"type": "string"},
        "risks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string"},
                    "severity": {
                        "type": "string",
                        "enum": ["low", "medium", "high", "critical"]
                    },
                    "mitigation": {"type": "string"},
                },
                "required": ["description", "severity"]
            }
        },
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "recommendation": {"type": "string"}
    },
    "required": ["project_status", "summary", "risks", "recommendation"]
}

answer = client.reflect(
    bank_id=BANK_ID,
    query="Perform a risk assessment of Project Phoenix.",
    response_schema=risk_schema,
    budget="high"
)

# The structured output is in answer.structured_output
if answer.structured_output:
    assessment = answer.structured_output
    print(f"Project Status: {assessment.get('project_status', 'N/A').upper()}")
    print(f"\nSummary: {assessment.get('summary', 'N/A')}")
    print(f"\nRisks:")
    for risk in assessment.get('risks', []):
        print(f"  [{risk.get('severity', '?').upper()}] {risk.get('description', '?')}")
        if risk.get('mitigation'):
            print(f"       Mitigation: {risk['mitigation']}")
    print(f"\nConfidence: {assessment.get('confidence', '?')}")
    print(f"Recommendation: {assessment.get('recommendation', 'N/A')}")
else:
    print("No structured output — check if schema is supported by your model.")
    print(f"Raw text:\n{answer.text}")

## 5. Context Parameter — Additional Instructions

The `context` parameter passes extra guidance to the reasoning LLM without affecting retrieval.

In [ ]:
# Same query, different context → different answers
query = "What should the team focus on next?"

# Context: executive summary
exec_answer = client.reflect(
    bank_id=BANK_ID,
    query=query,
    context="This is for an executive summary. Keep it high-level, focus on business impact."
)
print("=== Executive Context ===")
print(exec_answer.text[:300])

print("\n" + "="*50)

# Context: technical deep-dive
tech_answer = client.reflect(
    bank_id=BANK_ID,
    query=query,
    context="This is for the engineering team. Go into technical details, mention specific services and patterns."
)
print("=== Technical Context ===")
print(tech_answer.text[:300])

## 6. Disposition Traits — How Personality Changes Answers

Create banks with different dispositions and see how the same facts produce different reflections.

In [ ]:
# Create two banks with opposite dispositions
# Skeptical bank
client.create_bank(
    bank_id="tutorial-skeptical",
    name="Skeptical Analyst",
    reflect_mission="I am a critical analyst. I question assumptions and demand evidence.",
    disposition={"skepticism": 5, "literalism": 4, "empathy": 1}
)

# Empathetic bank
client.create_bank(
    bank_id="tutorial-empathetic",
    name="Supportive Coach",
    reflect_mission="I am a supportive coach. I consider team morale and individual well-being.",
    disposition={"skepticism": 1, "literalism": 2, "empathy": 5}
)

# Seed both with the same facts
shared_facts = [
    "Project is 3 weeks behind schedule",
    "Two senior engineers left the team in Q1",
    "Test coverage is only 45%",
    "Team is working overtime to catch up",
    "Deadline is September 2026 — CEO wants no delays",
]

for fact in shared_facts:
    client.retain(bank_id="tutorial-skeptical", content=fact)
    client.retain(bank_id="tutorial-empathetic", content=fact)

print("✓ Seeded both banks with identical facts")

In [ ]:
# Same question → different answers based on disposition
question = "Should we push the team harder to meet the September deadline?"

print("=== Skeptical Analyst (skepticism=5, empathy=1) ===")
skeptical = client.reflect(bank_id="tutorial-skeptical", query=question, budget="high")
print(skeptical.text[:300])

print("\n" + "="*60)

print("=== Supportive Coach (skepticism=1, empathy=5) ===")
empathetic = client.reflect(bank_id="tutorial-empathetic", query=question, budget="high")
print(empathetic.text[:300])

## 7. Directives — Hard Guardrails

Directives are rules that must never be violated, unlike disposition (which is soft influence).

In [ ]:
# Create a bank with strict directives
client.create_bank(
    bank_id="tutorial-compliant",
    name="Compliant Assistant",
    reflect_mission="I am a financial information assistant.",
    directives=[
        "Never make specific investment recommendations.",
        "Never predict future stock prices or market movements.",
        "Always state that past performance does not guarantee future results.",
        "Always cite the source and date of any financial data."
    ]
)

# Seed with some financial info
client.retain(bank_id="tutorial-compliant",
    content="Company XYZ's stock was $150 on June 1, 2026. Revenue grew 20% YoY.")
client.retain(bank_id="tutorial-compliant",
    content="The user is interested in tech stocks and has a high risk tolerance.")

# Try to get investment advice — directives should block it
answer = client.reflect(
    bank_id="tutorial-compliant",
    query="Should I buy XYZ stock right now? I think it will go up 30% next month."
)
print("=== Response (with directives) ===")
print(answer.text[:400])

## 8. Practical Pattern: Decision Support Agent

In [ ]:
class DecisionSupportAgent:
    """Uses Hindsight to provide evidence-backed decision support."""
    
    def __init__(self, client, bank_id):
        self.client = client
        self.bank_id = bank_id
    
    def analyze_decision(self, question, options=None):
        """Analyze a decision and provide a structured recommendation."""
        
        context_parts = ["Analyze the pros and cons based on all available evidence."]
        if options:
            context_parts.append(f"Consider these options: {', '.join(options)}")
        
        answer = self.client.reflect(
            bank_id=self.bank_id,
            query=question,
            context=" ".join(context_parts),
            budget="high",
            response_schema={
                "type": "object",
                "properties": {
                    "recommendation": {"type": "string"},
                    "pros": {"type": "array", "items": {"type": "string"}},
                    "cons": {"type": "array", "items": {"type": "string"}},
                    "confidence": {"type": "number"},
                    "alternatives": {"type": "array", "items": {"type": "string"}},
                }
            }
        )
        
        if answer.structured_output:
            return answer.structured_output, answer.based_on
        return {"recommendation": answer.text}, answer.based_on

# Usage
advisor = DecisionSupportAgent(client, BANK_ID)
result, evidence = advisor.analyze_decision(
    "Should we hire more engineers or extend the timeline?",
    options=["Hire 3 more engineers", "Extend timeline by 2 months", "Reduce scope"]
)
print(json.dumps(result, indent=2))
print(f"\nBased on {len(evidence)} memories")

## 9. Cleanup

In [ ]:
# Clean up temporary banks
# client.delete_bank(bank_id="tutorial-skeptical")
# client.delete_bank(bank_id="tutorial-empathetic")
# client.delete_bank(bank_id="tutorial-compliant")
# client.delete_bank(bank_id=BANK_ID)
print("Banks preserved.")

## Summary

You learned:
1. **Basic reflect** — ask questions, get synthesized answers
2. **Evidence tracking** — `based_on` traces claims to source facts
3. **Budget levels** — controlling analysis depth
4. **Structured output** — `response_schema` for JSON responses
5. **Context parameter** — additional guidance without affecting retrieval
6. **Disposition traits** — how skepticism/empathy/literalism shape answers
7. **Directives** — hard guardrails that cannot be violated

**Next:** [06_banks.ipynb](06_banks.ipynb) — bank management and configuration